# UOR-R4 Geometric Model Training on Google Colab (CPU - Option B)

This notebook trains the **R4 Geometric Language Model (Option B: 8M parameters)** directly on **Google Colab CPU** using Mathlib 4 formal proof tactics extracted via LeanDojo.

- **Runtime:** Standard Google Colab CPU (Free tier, zero GPU required, zero load on your local PC).
- **Architecture:** UOR-R4 4D Clifford / Quaternion Geometric Stack (Width=288, Heads=6, Layers=6, Pattern=`rrarra`).
- **Task:** Next-tactic generation for Lean 4 theorem proving.
- **Output:** Model checkpoints and safetensors automatically saved to Google Drive or local Colab storage.

### 1. Inspect Colab CPU Environment & Mount Google Drive
Mounting Google Drive is recommended so your checkpoints (`/content/drive/MyDrive/r4_checkpoints/...`) persist across Colab sessions.

In [ ]:
# Inspect CPU specs
!lscpu | grep -E 'Model name|CPU\(s\):|Thread|Core'
!free -h

# Optional: Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

### 2. Clone the syn-prover Repository
Clones the repository and checks out the `leanDojo` branch.

In [ ]:
%cd /content
!rm -rf /content/syn-prover
!git clone -b leanDojo https://github.com/wwwportal/prover.git syn-prover
%cd /content/syn-prover

### 3. Bootstrap Environment (Rust + Python Requirements)
Installs the Rust toolchain, PyTorch, Tokenizers, SafeTensors, and compiles the `geometric-stack` binary optimized for CPU.

In [ ]:
!bash scripts/colab_bootstrap.sh

### 4. Fetch & Tokenize Mathlib 4 Dataset
Downloads the Lean 4 Mathlib benchmark dataset and generates the ByteLevel BPE token streams (`train.u16`, `valid.u16`) and response masks.

In [ ]:
# Download the Mathlib4 benchmark JSONs from Hugging Face
!bash scripts/fetch_mathlib4_dataset.sh

# Tokenize state-tactic pairs into binary streams for R4
!python3 scripts/prepare_mathlib_for_r4.py

### 5. Launch Option B Training (8M Model on CPU)
Launches the R4 geometric training loop on the CPU. Checkpoints will be saved automatically to Google Drive (if mounted) or `reports/`.

In [ ]:
import os

# Option B hyperparameters
os.environ['MODEL_SIZE'] = '8m'         # 8M parameter geometric model
os.environ['DEVICE'] = 'cpu'            # Run on CPU
os.environ['STEPS'] = '3000'            # Fast CPU run (~15 min); increase to 7324 for full pass
os.environ['BATCH'] = '8'               # CPU-friendly mini-batch
os.environ['CONTEXT'] = '256'           # Context window
os.environ['EVAL_EVERY'] = '250'
os.environ['CHECKPOINT_EVERY'] = '500'

!bash scripts/colab_train.sh

### 6. Evaluate Tactic Prediction & Perplexity
Tests the trained checkpoint on Lean 4 benchmark theorems, measuring cross-entropy loss, perplexity, and geometric quaternion drift.

In [ ]:
# Tactic likelihood and perplexity evaluation
!python3 src/eval/tactic_eval.py

# Geometric manifold / quaternion transport diagnostics
!python3 src/eval/r4_runner.py

### 7. Interactive Step-by-Step Tactic Generation (LeanDojo Adapter)
Demonstrates the R4 prover responding to dynamic Lean goal states with single-step tactic suggestions as expected by LeanDojo.

In [ ]:
!python3 src/eval/r4_prover.py